# A tokenizer from scratch

Run the cells from top to bottom. After that, you can edit and rerun any demonstration cell separately. The complete flow is: **raw text -> tokens -> vocabulary -> token IDs -> decoded text**.

## 1. Define a small training corpus

In [1]:
import re

TRAINING_CORPUS = """Hello, world!
Mira asked, \"Are you ready?\"
The answer was simple: yes; we can begin.
A small tokenizer -- built from scratch -- turns text into tokens.
(Every token has an ID.)"""

print(TRAINING_CORPUS)

Hello, world!
Mira asked, "Are you ready?"
The answer was simple: yes; we can begin.
A small tokenizer -- built from scratch -- turns text into tokens.
(Every token has an ID.)


## 2. Split text into tokens

The regular expression checks for special tokens and `--` first. It then finds words, common punctuation, and any remaining non-whitespace symbol. Because whitespace is never matched, there are no whitespace-only tokens.

In [2]:
TOKEN_PATTERN = re.compile(
    r"<\|[^|]+\|>|--|\w+(?:'\w+)?|[.,?!:;\"()]|[^\w\s]"
)

def tokenize(text):
    """Turn text into words and separate punctuation tokens."""
    return TOKEN_PATTERN.findall(text)

example_text = "Hello, world!"
print("Raw text:", example_text)
print("Tokens:  ", tokenize(example_text))

Raw text: Hello, world!
Tokens:   ['Hello', ',', 'world', '!']


## 3. Join tokens into readable text

A plain join would produce `Hello , world !`. This small helper removes obvious extra spaces around punctuation, parentheses, and paired quotation marks.

In [3]:
def tokens_to_text(tokens):
    """Join tokens and repair the most obvious punctuation spacing."""
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,?!:;)])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    text = re.sub(r'"\s*(.*?)\s*"', r'"\1"', text)
    return text

print(tokens_to_text(["Hello", ",", "world", "!"]))

Hello, world!


## 4. Build a deterministic vocabulary

A vocabulary contains each unique training token once. Sorting makes the assigned IDs deterministic. **The IDs are only identifiers; their numeric values contain no semantic meaning.**

In [4]:
corpus_tokens = tokenize(TRAINING_CORPUS)
vocabulary = sorted(set(corpus_tokens))

token_to_id = {token: token_id for token_id, token in enumerate(vocabulary)}
id_to_token = {token_id: token for token, token_id in token_to_id.items()}

print("Corpus tokens:", corpus_tokens)
print("Vocabulary size:", len(vocabulary))
print("token_to_id:", token_to_id)
print("id_to_token:", id_to_token)

Corpus tokens: ['Hello', ',', 'world', '!', 'Mira', 'asked', ',', '"', 'Are', 'you', 'ready', '?', '"', 'The', 'answer', 'was', 'simple', ':', 'yes', ';', 'we', 'can', 'begin', '.', 'A', 'small', 'tokenizer', '--', 'built', 'from', 'scratch', '--', 'turns', 'text', 'into', 'tokens', '.', '(', 'Every', 'token', 'has', 'an', 'ID', '.', ')']
Vocabulary size: 40
token_to_id: {'!': 0, '"': 1, '(': 2, ')': 3, ',': 4, '--': 5, '.': 6, ':': 7, ';': 8, '?': 9, 'A': 10, 'Are': 11, 'Every': 12, 'Hello': 13, 'ID': 14, 'Mira': 15, 'The': 16, 'an': 17, 'answer': 18, 'asked': 19, 'begin': 20, 'built': 21, 'can': 22, 'from': 23, 'has': 24, 'into': 25, 'ready': 26, 'scratch': 27, 'simple': 28, 'small': 29, 'text': 30, 'token': 31, 'tokenizer': 32, 'tokens': 33, 'turns': 34, 'was': 35, 'we': 36, 'world': 37, 'yes': 38, 'you': 39}
id_to_token: {0: '!', 1: '"', 2: '(', 3: ')', 4: ',', 5: '--', 6: '.', 7: ':', 8: ';', 9: '?', 10: 'A', 11: 'Are', 12: 'Every', 13: 'Hello', 14: 'ID', 15: 'Mira', 16: 'The', 17

## 5. SimpleTokenizerV1

In [5]:
class SimpleTokenizerV1:
    """A tokenizer that only understands tokens in its vocabulary."""

    def __init__(self, token_to_id):
        self.token_to_id = token_to_id
        self.id_to_token = {
            token_id: token for token, token_id in token_to_id.items()
        }

    def encode(self, text):
        tokens = tokenize(text)
        # A missing token raises KeyError. V1 has no fallback.
        return [self.token_to_id[token] for token in tokens]

    def decode(self, ids):
        tokens = [self.id_to_token[token_id] for token_id in ids]
        return tokens_to_text(tokens)

## 6. Encode and decode known text

In [6]:
tokenizer_v1 = SimpleTokenizerV1(token_to_id)

known_text = "Hello, world!"
known_ids = tokenizer_v1.encode(known_text)

print("Original:", known_text)
print("Tokens:  ", tokenize(known_text))
print("IDs:     ", known_ids)
print("Decoded: ", tokenizer_v1.decode(known_ids))

Original: Hello, world!
Tokens:   ['Hello', ',', 'world', '!']
IDs:      [13, 4, 37, 0]
Decoded:  Hello, world!


## 7. Intentionally demonstrate V1's unknown-word problem

`galaxy` did not occur in the training corpus, so it has no ID. The `try`/`except` lets the notebook display the expected failure without stopping later cells.

In [9]:
unknown_text = "Hello, galaxy!"
print("Trying to encode:", unknown_text)

try:
    tokenizer_v1.encode(unknown_text)
except KeyError as error:
    print(f"V1 failed as expected: {error.args[0]!r} is not in the vocabulary.")

Trying to encode: Hello, galaxy!
V1 failed as expected: 'galaxy' is not in the vocabulary.


## 8. Add `<|unk|>` and `<|endoftext|>`

Copying the V1 mapping keeps all existing IDs unchanged. The two special tokens receive the next two fixed IDs.

In [10]:
v2_token_to_id = dict(token_to_id)
v2_token_to_id["<|unk|>"] = len(v2_token_to_id)
v2_token_to_id["<|endoftext|>"] = len(v2_token_to_id)

print("<|unk|> ID:      ", v2_token_to_id["<|unk|>"])
print("<|endoftext|> ID:", v2_token_to_id["<|endoftext|>"])

<|unk|> ID:       40
<|endoftext|> ID: 41


## 9. SimpleTokenizerV2

In [11]:
class SimpleTokenizerV2:
    """A tokenizer that maps unseen tokens to <|unk|>."""

    def __init__(self, token_to_id):
        self.token_to_id = token_to_id
        self.id_to_token = {
            token_id: token for token, token_id in token_to_id.items()
        }
        self.unknown_id = token_to_id["<|unk|>"]

    def encode(self, text):
        tokens = tokenize(text)
        return [
            self.token_to_id.get(token, self.unknown_id) for token in tokens
        ]

    def decode(self, ids):
        tokens = [self.id_to_token[token_id] for token_id in ids]
        return tokens_to_text(tokens)

## 10. Test V2 with unknown words and two documents

The first and second document are separated by `<|endoftext|>`. Unseen words such as `galaxy`, `robot`, and `dances` all use the same `<|unk|>` ID, so their original spelling cannot be recovered during decoding.

In [12]:
tokenizer_v2 = SimpleTokenizerV2(v2_token_to_id)

multi_document_text = "Hello, galaxy! <|endoftext|> A small robot dances."
multi_document_ids = tokenizer_v2.encode(multi_document_text)

print("Original:", multi_document_text)
print("Tokens:  ", tokenize(multi_document_text))
print("IDs:     ", multi_document_ids)
print("Decoded: ", tokenizer_v2.decode(multi_document_ids))

Original: Hello, galaxy! <|endoftext|> A small robot dances.
Tokens:   ['Hello', ',', 'galaxy', '!', '<|endoftext|>', 'A', 'small', 'robot', 'dances', '.']
IDs:      [13, 4, 40, 0, 41, 10, 29, 40, 40, 6]
Decoded:  Hello, <|unk|>! <|endoftext|> A small <|unk|> <|unk|>.


## 11. BOS, EOS, and PAD concepts

- **BOS** marks the beginning of a sequence.
- **EOS** marks the end of a sequence.
- **PAD** fills shorter sequences so several sequences can have equal lengths in a batch.

These three tokens are conceptual in this exercise; they are not added to the V2 vocabulary.

In [13]:
short_sequence = ["<|bos|>", "Hello", "!", "<|eos|>"]
long_sequence = ["<|bos|>", "Are", "you", "ready", "?", "<|eos|>"]

target_length = max(len(short_sequence), len(long_sequence))
padded_short = short_sequence + ["<|pad|>"] * (target_length - len(short_sequence))
padded_long = long_sequence + ["<|pad|>"] * (target_length - len(long_sequence))

print("Padded short sequence:", padded_short)
print("Padded long sequence: ", padded_long)

Padded short sequence: ['<|bos|>', 'Hello', '!', '<|eos|>', '<|pad|>', '<|pad|>']
Padded long sequence:  ['<|bos|>', 'Are', 'you', 'ready', '?', '<|eos|>']


## 12. What BPE will solve next

This tokenizer is intentionally educational. Real GPT-style tokenizers do not normally solve unfamiliar text with a single `<|unk|>` token. **Byte Pair Encoding (BPE)** and other subword methods can split an unfamiliar word into smaller known pieces. A byte-level tokenizer can ultimately represent arbitrary text. BPE is deliberately left for the next exercise.